##RAG - RETRIVAL AUGUMENTED GENERATION

-Rag is used to minimize the hallucination ,
-the data is stored(chromDB => converts into embedding) in rag system

 - rag is like an search engine is used to get the informations from the chromaDB so it gives only know values without hallucination


##hallucination problem
   ->In the context of Retrieval Augmented Generation (RAG), hallucination refers to the generation of plausible-sounding but factually incorrect, unsupported, or contradictory information by the language model, even when provided with relevant external knowledge or documents. It's when the model 'makes things up' rather than strictly adhering to the provided facts.
(Plausible means seeming reasonable, probable, or likely to be true)



RAG Architecture: Pipeline:

Phase A: Ingestion
1. Documents
2. Chunks
3. Embed
4. Store

Phase B: Inference
1. User Qn
2. Embed Qn

In [ ]:
## Install requirements
##sentence-transformers is used to convert string to integer
# groq is used to call LLM
# chromaDb is used to store the vectors
#pandas is used for fetching the data
!pip install sentence-transformers chromadb groq pandas -q

print(" imported succesfully")

In [ ]:
#after install we import every library

import pandas as pd

import chromadb
#it is an vector storing library

from sentence_transformers import SentenceTransformer
#SentenceTransformer: a class that loads an pre trained  embedding model

from groq import Groq
# the Groq API is used to call the LLM

import os
# os: python's buildin library for interacting with os

In [ ]:
GROQ_API_KEY="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"

os.environ['GROQ_API_KEY'] = GROQ_API_KEY  ## it is like an locker for key hiding

groq_client=Groq(api_key=GROQ_API_KEY)  # initalizes the groq client ,groq_client is our connection to the groq API Key

print("GROQ API key is initialized")
print("Note: If you are an authentication error later,double check your API key")

In [ ]:
df=pd.read_csv('/content/drive/MyDrive/Copy of college_notes.csv')
print("Shape of dataset:",df.shape)
print("\nColumns:",df.columns.tolist())
print("\nFirst 3 rows")
print(df.head(3))



In [ ]:
print("Subject in the dataset:")

print(df['subject'].value_counts())

print('\nSample pf topics:')
print(df[['note_id','subject','topic']].to_string(index=False))
print("\nLength od content(number of characters) for each note:")
df['content_length']=df['content'].apply(len)
print(df[['topic','content_length']].to_string(index=False))

In [ ]:
documents=df['content'].tolist()

ids=[f"note_{row['note_id']}" for row in df.to_dict('records')]

metadata = [
    {'subject': row['subject'],'topic':row['topic']}
    for row in df.to_dict('records')
]
print(f"Total chunks prepared:{len(documents)}")
print(f"First document ID :{ids[0]}")
print(f"First metadata:{metadata[0]}")
print(f"First Document: {documents[0][:100]}...")

In [ ]:
print("Loading embedding model...")
print("(This my take 30-60 seconds on first run -model is downloaded)")
print("Subsequent runs will be faster as the model is cached")

embedding_model=SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding model loaded")

test_embedding = embedding_model.encode("This is an text sentence")

print(f"\nTest embedding shape:{test_embedding.shape}")

print(f"\nFirst 5 values of test embedding:{test_embedding[:5]}")
## we lose the first 5 values to see what an embedding looks like

In [ ]:
### chromaDB stores the data

chroma_client = chromadb.Client()
#using the chromadb clinet we store the data in memory
#in memory means ,data exists while the running in notebook
# for permanent storage you would use chromadb.PersistantClient('path'./chromadb)


collection =chroma_client.get_or_create_collection(name="college_notes_rag")
# a collection is chromaDb is like an table in regular db
# it groups related documents
#get_or collection():cretes an if it is not existing one of it

print("ChromaDb client created ")
print("Collection name:college_notes_rag")
print(f"Documents in collection so far:{collection.count()}")

In [ ]:
print("Generating embedding for all 15 notes...")
print("This my task 15-30 seconds")

embeddings = embedding_model.encode(documents,show_progress_bar=True)
print(f"Embedding matrix shape:{embeddings.shape}")
embeddings_list = embeddings.tolist()

collection.add(
    documents=documents, # the actual text contest of each node
    ids=ids, # unique string IDS
    metadatas=metadata,  #subject and topic infor for each node
    embeddings=embeddings_list  # the vector representation
)

In [ ]:
def retrieve_relevant_chunks(question, top_k=3):
  """
  Given a user question, retrieve the most relevant document chunks form ChromaDB.

  Parameters:
      question (str) : The user's questions as a text string
      top_k     (int) : How many top results to return (default: 3)

  Returns:
      A dictionary containing the retrived documents, distances and their metadata
  """
  question_embedding = embedding_model.encode(question).tolist()
  results = collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k
  )

  return results

print("Retrival function defined successfully!")
print("Function: retrieve_relevant_chunks(question, top_k=3)")

In [ ]:
## call
## why we use metadata becasue don't needs to check the embeddings always better to check the particular data
test_question="What is ETL"
results = retrieve_relevant_chunks(test_question, top_k=3)
print("\nTop 3 Retrieved Chunks:")
print('-'*30)

for i, (doc, dist, meta) in enumerate(zip(results['documents'][0], results['distances'][0], results['metadatas'][0])):
  print(f"Document {i+1}")
  print(f"\nResult {i+1}:")
  print(f"  Subject   : {meta['subject']}")
  print(f"  Topic    : {meta['topic']}")
  print(f"  Distance : {dist}")
  print(f"  Content  : {doc[:100]}...")

In [ ]:
def build_context_from_results(results):
  context_parts = []

  for i, (doc, meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):
    context_text = f"[Source {i+1}: {meta['subject']} - {meta['topic']}\n{doc}]"
    context_parts.append(context_text)

    return context_parts

In [ ]:
# one chunck = it has 200-500 words

 # the model input is context and with the input what are the possibile questions asked by the user
def generate_rag_element(question, context):
  """
  Send the restricted context and question to the Groq LLM for answer generation

  Parameters:
    questions(str) : The user's question
    context (str) : The restricted context chunks(formatted string)

  Returns:
    answer(str) : the LLM's generated answer

  """

  # SYSTEM PROMPT : Instruction to the LLM about its role and behaviour
  #This is the key to RAG -we tell the LLM to ONLY use the provided context (i don't know word is restricted in all LLM'S)

  system_prompt = """You are a helpful academic assistant for engineering students

  You will be given an context retrieved from a college knowledge base, and a student's question

  RULES:
  1.Answer only using the instruction provided in the context below
  2.If the answer is not found in the context, say exactly:
      "I don't have enough information in my knowledge base to answer this question."
  3.Do not use your general traning knowledge
  4.keep answer clear,accurate,and begineer-friendly
  5.Mention which source the information came from when possible."""

  #USER PROMPT: the context+ question formatted

  user_prompt=f"""Context:
  {context}

  Question:
  {question}
  Please answer the question based only on the context provided above ."""

  response = groq_client.chat.completions.create(
      model="gpt-3.5-turbo", # llama-3.1.8b-instant-to use same model from traning
      messages=[
          {"role":"system","content":system_prompt},
          {"role":"user","content":user}
      ],
      temperature = 0.1,
#temperature - 0.1 - Very low randomness - we want factual, consistent answers, for RAG, low temp is preffered so the LLM sticks to the context
      max_tokens = 500 #max len of gen responce
  )

  #Extract the text answer from the API resonse object
  answer = response.choices[0].message.content
  #response.choices : A list of responses
  return answer
print("Defined RAG generation function")

In [ ]:
def ask_college_assistant(question, top_k=3, verbose=True):
    """
    Parameters:
      question (str) : The user's question
      top_k    (int): Returns the top 3 chunks from the collection
      verbose (bool): Whether to print intermediate steps (True)

    Returns:
      answer (str): The final generated answer
    """
    if verbose:
      print(f"Question: {question}")
      print("="*60)
      print("Step 1: Retrieving Relevant Chunks...")
      print("="*60)

    results = retrieve_relevent_chunks(question, top_k=top_k)

    if verbose:
      print("Step 2: Building Context...")
      print("="*60)
    context = build_context_from_results(results)

    if verbose:
      print("Step 3: Generating Answer...")
      print("="*60)
    answer = generate_rag_answer(question, context)

    return answer